# Nexus workbench quickstart

This notebook runs **inside the platform network** — every service is one hostname away, and all credentials are already in the environment:

| Service | Env var already set |
| --- | --- |
| Control plane (gateway) | `MLAIOPS_URL` |
| MLflow tracking + registry | `MLFLOW_TRACKING_URI` |
| MinIO (S3 artifacts) | `MLFLOW_S3_ENDPOINT_URL`, `AWS_*` |
| Feature gateway (online store) | `MLAIOPS_FEATURE_GATEWAY_URL` |
| Kafka REST | `KAFKA_REST_URL` |
| Prefect | `PREFECT_API_URL` |
| Postgres (+pgvector) | `DATABASE_URL` |
| Langfuse | `LANGFUSE_HOST`, `LANGFUSE_*_KEY` |

There is also a **terminal** (File → New → Terminal) with the same environment — `git`, `curl`, and all platform modules are available.


## 1. Who am I, and is the platform healthy?

In [ ]:
import os, httpx

gateway = os.environ["MLAIOPS_URL"]
print(httpx.get(f"{gateway}/api/v1/health").json())
print(httpx.get(f"{gateway}/api/v1/me").json())

## 2. Browse what the platform already knows
Projects, registered models and applied feature views — the same data the console shows.

In [ ]:
projects = httpx.get(f"{gateway}/api/v1/projects").json()
models = httpx.get(f"{gateway}/api/v1/models").json()["items"]
features = httpx.get(f"{gateway}/api/v1/features").json()["items"]
print(f"projects: {[p['name'] for p in projects]}")
print(f"models:   {[(m['name'], m['version'], m['deployment_status']) for m in models]}")
print(f"features: {[f['name'] for f in features]}")

## 3. Train a model and log it to MLflow
The workbench pins `scikit-learn==1.7.0` and `mlflow==3.1.1` — the exact versions the serving containers run, so anything you log here serves cleanly.

In [ ]:
import mlflow
from sklearn.datasets import make_classification
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score

X, y = make_classification(n_samples=1200, n_features=12, n_informative=6,
                           n_clusters_per_class=1, class_sep=1.5, random_state=7)
X_train, X_test, y_train, y_test = train_test_split(X, y, random_state=7)
model = LogisticRegression(max_iter=500).fit(X_train, y_train)
accuracy = accuracy_score(y_test, model.predict(X_test))

mlflow.set_experiment("workbench")
with mlflow.start_run() as run:
    mlflow.log_metric("accuracy", accuracy)
    logged = mlflow.sklearn.log_model(model, name="model",
                                      registered_model_name="notebook-classifier")
print(f"accuracy={accuracy:.3f}  model_uri={logged.model_uri}")

## 4. Register it with the control plane
This puts the model in the console's registry (Models tab) with its quality gate, and makes it deployable to a live endpoint from there.

In [ ]:
registered = mlflow.MlflowClient().get_latest_versions("notebook-classifier")[0]
payload = {
    "project_id": projects[0]["id"] if projects else "",
    "name": "notebook-classifier",
    "version": registered.version,
    "artifact_uri": logged.model_uri,
    "metrics": {"accuracy": accuracy},
}
response = httpx.post(f"{gateway}/api/v1/models", json=payload)
print(response.status_code, response.json())

## 5. Read online features (Redis, via the feature gateway)

In [ ]:
feature_gateway = os.environ["MLAIOPS_FEATURE_GATEWAY_URL"]
lookup = httpx.post(f"{feature_gateway}/get-online-features",
                    json={"feature_service": "customer_profile",
                          "entities": [{"entity_id": "u123"}]})
print(lookup.json())

## 6. Produce real-time events onto Kafka
The realtime processor consumes these, enriches them with online features, scores them, and publishes alerts — watch the Real-Time tab in the console while this runs.

In [ ]:
!python -m realtime.produce --demo fraud --count 3

import time; time.sleep(10)  # the stream processor consumes, scores and reports
print(httpx.get(f"{gateway}/api/v1/realtime").json())

## 7. Where to go next
- **Console:** [http://localhost:8080](http://localhost:8080) — deploy the model you just registered, then hit it from the predict console.
- **MLflow UI:** [http://localhost:15000](http://localhost:15000) — the run and registered model from step 3.
- **Prefect UI:** [http://localhost:4200](http://localhost:4200) — flow runs from the Pipelines tab.
- **Langfuse:** [http://localhost:3000](http://localhost:3000) — agent traces (login: `admin@local.dev` / `mlaiops-local-admin`).
- **Terminal:** File → New → Terminal, then e.g. `python -m realtime.produce --demo callcenter --count 2`
- The full connection reference lives in `docs/connecting-services.md` in the repo.
